# Depthwise Separable Convolution（深度可分离卷积）

**难度：** Medium

**函数签名：** `depthwise_separable_conv(x, dw_weight, pw_weight) -> Tensor`

**参数：**
- `x` — 输入张量 (B, C_in, H, W)
- `dw_weight` — 深度滤波器 (C_in, 1, kH, kW)
- `pw_weight` — 逐点滤波器 (C_out, C_in, 1, 1)

**返回：** 输出张量 (B, C_out, H-kH+1, W-kW+1)

**要求：**
- 无填充，步长=1
- 不得使用 `F.conv2d`——使用 `unfold` 和 `einsum`/matmul 实现
- 深度步骤：通道 c 仅与 `dw_weight[c, 0]` 卷积
- 逐点步骤：1x1 卷积通过矩阵乘法混合通道

**提示：** 深度卷积用 `unfold` 提取滑窗 patches，逐点卷积用 `einsum` 混合通道

In [ ]:
import torch
import math

In [ ]:
# ✅ SOLUTION

def depthwise_separable_conv(x, dw_weight, pw_weight):
    # ---- Step 1: 解包输入形状 ----
    # x 的形状: (B, C_in, H, W)
    # B=批大小, C_in=输入通道数, H=高度, W=宽度
    B, C_in, H, W = x.shape
    # dw_weight 的形状: (C_in, 1, kH, kW)
    # 注意: 深度卷积每个通道独立卷积，所以第2维是1
    kH, kW = dw_weight.shape[2], dw_weight.shape[3]

    # ---- Step 2: 用 unfold 提取滑窗 patches ----
    # unfold(dim, size, step) 在指定维度上滑动窗口
    # x.unfold(2, kH, 1): 在 H 维度上以窗口大小 kH、步长 1 滑动
    #   原始: (B, C_in, H, W) → unfold后: (B, C_in, H-kH+1, W, kH)
    # x.unfold(3, kW, 1): 再在 W 维度上以窗口大小 kW、步长 1 滑动
    #   → 最终: (B, C_in, H_out, W_out, kH, kW)
    # 其中 H_out = H-kH+1, W_out = W-kW+1
    # 每个 patches[b, c, i, j] 就是通道 c 在位置 (i,j) 处的 kH×kW 局部窗口
    patches = x.unfold(2, kH, 1).unfold(3, kW, 1)

    # ---- Step 3: 深度卷积 — 逐通道卷积 ----
    # dw_weight[:, 0] 的形状: (C_in, kH, kW) — 去掉多余的第2维
    # .view(1, C_in, 1, 1, kH, kW) — 重塑为可与 patches 广播的形状
    #   第0维 1: 广播到 B 个样本
    #   第1维 C_in: 对应每个输入通道
    #   第2维 1: 广播到 H_out 个位置
    #   第3维 1: 广播到 W_out 个位置
    #   第4,5维 kH,kW: 卷积核的空间尺寸
    # patches * weight: 逐元素相乘（每个通道独立）
    # .sum(dim=(-2, -1)): 对 kH×kW 求和，完成卷积
    # 结果 dw_out 形状: (B, C_in, H_out, W_out)
    dw_out = (patches * dw_weight[:, 0].view(1, C_in, 1, 1, kH, kW)).sum(dim=(-2, -1))

    # ---- Step 4: 逐点卷积 — 1×1 卷积混合通道 ----
    # pw_weight[:, :, 0, 0] 形状: (C_out, C_in) — 提取1×1卷积核的权重矩阵
    # einsum('bchw,oc->bohw') 含义:
    #   b=batch, c=input_channel, h=height, w=width
    #   o=output_channel, c=input_channel (与上面的c求和)
    # 这等价于对每个空间位置做矩阵乘法: out = W @ dw_out
    # 结果形状: (B, C_out, H_out, W_out)
    out = torch.einsum('bchw,oc->bohw', dw_out, pw_weight[:, :, 0, 0])
    return out

In [ ]:
torch.manual_seed(0)
B, C_in, H, W = 2, 4, 8, 8
C_out, kH, kW = 8, 3, 3

x         = torch.randn(B, C_in, H, W)
dw_weight = torch.randn(C_in, 1, kH, kW)   # one kernel per input channel
pw_weight = torch.randn(C_out, C_in, 1, 1)  # 1x1 conv

out = depthwise_separable_conv(x, dw_weight, pw_weight)
print("Output shape:", out.shape)  # (2, 8, 6, 6)

# Verify channel independence of depthwise step
patches = x.unfold(2, kH, 1).unfold(3, kW, 1)
dw_ch0 = (patches[:, 0:1] * dw_weight[0:1, 0].view(1, 1, 1, 1, kH, kW)).sum(dim=(-2, -1))
dw_ch1 = (patches[:, 1:2] * dw_weight[1:2, 0].view(1, 1, 1, 1, kH, kW)).sum(dim=(-2, -1))
print("DW ch0 and ch1 are independent (cross-correlation ~0):",
      (dw_ch0 * dw_ch1).mean().abs().item() < 1.0)

In [ ]:
from torch_judge import check
check("depthwise_conv")

## 📝 核心思路总结

1. **深度可分离卷积 = 深度卷积 + 逐点卷积**，将标准卷积分解为两步，大幅减少参数量
2. **深度卷积**：用 `unfold` 提取滑窗 patches，每个通道独立卷积（通道间无交互）
3. **逐点卷积**：本质是 1×1 卷积，用 `einsum` 或矩阵乘法混合通道信息
4. **关键技巧**：`unfold` 实现 im2col，`view` 调整形状实现广播乘法